In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

e:\langraph\myvenv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
load_dotenv()

True

In [3]:
model = ChatGroq(
    model="llama-3.1-8b-instant"  # free model
    
)

In [5]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [6]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = model.invoke(prompt).content

    return {'joke': response}

In [7]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = model.invoke(prompt).content

    return {'explanation': response}

In [8]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()#createing checkpointer object

workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to therapy? \n\nBecause it was feeling a little crusty.',
 'explanation': 'The joke "Why did the pizza go to therapy? Because it was feeling a little crusty" is a play on words. Here\'s a breakdown of the explanation:\n\n1. The setup of the joke asks the listener to consider why a pizza would go to therapy. This sets the stage for a possible answer, hinting that there might be something unusual or humorous about the reason.\n\n2. The punchline "Because it was feeling a little crusty" is where the joke lies. The word "crusty" has a double meaning here:\n   - In the context of pizza, a crust is the outer layer of the pizza, often crispy and crunchy. This is a literal meaning of the word.\n   - However, "feeling a little crusty" is also a common idiomatic expression used to describe someone who is irritable, grumpy, or feeling a bit down. This is a figurative meaning of the word.\n\n3. The joke relies on the listener\'s recognition of the 

In [12]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti go to therapy? \n\nBecause it was feeling a little "twisted" and had a lot of "noodle" issues.',
 'explanation': 'This joke is a play on words, which is a common technique used in humor. It takes advantage of multiple meanings and associations in the English language to create a pun.\n\nIn this case, the joke relies on the following wordplay:\n\n1. "Twisted": This phrase has a double meaning. It can refer to emotional distress or a state of being mentally unwell (as in, "feeling twisted"), but it can also refer to a twisted or tangled shape, which is a common characteristic of overcooked spaghetti.\n\n2. "Noodle issues": This phrase is another play on words. In a psychological context, "issues" typically refers to personal problems or emotional challenges. However, "noodle" is a colloquial term for spaghetti, making it a clever connection to the food in question.\n\nThe joke relies on the listener\'s understanding of these multiple mea

In [13]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti go to therapy? \n\nBecause it was feeling a little "twisted" and had a lot of "noodle" issues.', 'explanation': 'This joke is a play on words, which is a common technique used in humor. It takes advantage of multiple meanings and associations in the English language to create a pun.\n\nIn this case, the joke relies on the following wordplay:\n\n1. "Twisted": This phrase has a double meaning. It can refer to emotional distress or a state of being mentally unwell (as in, "feeling twisted"), but it can also refer to a twisted or tangled shape, which is a common characteristic of overcooked spaghetti.\n\n2. "Noodle issues": This phrase is another play on words. In a psychological context, "issues" typically refers to personal problems or emotional challenges. However, "noodle" is a colloquial term for spaghetti, making it a clever connection to the food in question.\n\nThe joke relies on the listener\'s understanding of

In [14]:
list(workflow.get_state_history(config2))#all intermediate states in the workflow execution

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti go to therapy? \n\nBecause it was feeling a little "twisted" and had a lot of "noodle" issues.', 'explanation': 'This joke is a play on words, which is a common technique used in humor. It takes advantage of multiple meanings and associations in the English language to create a pun.\n\nIn this case, the joke relies on the following wordplay:\n\n1. "Twisted": This phrase has a double meaning. It can refer to emotional distress or a state of being mentally unwell (as in, "feeling twisted"), but it can also refer to a twisted or tangled shape, which is a common characteristic of overcooked spaghetti.\n\n2. "Noodle issues": This phrase is another play on words. In a psychological context, "issues" typically refers to personal problems or emotional challenges. However, "noodle" is a colloquial term for spaghetti, making it a clever connection to the food in question.\n\nThe joke relies on the listener\'s understanding o